# Task 5
If you can't view this notebook locally, it can also be viewed [here](https://github.com/villekjellqvist/Matrix-Algebra-Python/blob/8110b30aa9938c21e4e3edae43cfacf1d0b0020d/HW3/HW3_Ville_Kjellqvist_Task5.ipynb).

In [ ]:
# Imports
import numpy as np
import numpy.typing as npt
from IPython.display import display

MSIZE = 5 # Define your matrix size here
SEED = 0 # Random seed for populating the matrix.

# Some intialization
rand = np.random.default_rng(SEED)

In this task, we will implement Jacobi's method for computing eigenvalues of a symmetrical matrix.
$A=[a_{ij}]\in M_n$ is a symmetric but not diagonal matrix. $U(i,j,\theta)$ is a Givens rotation matrix (which we will shorten to $U$ in the sequel),
and $B=[b_{pq}]=U^T(i,j,\theta)AU(i,j,\theta)$. Let $\theta$ be the solution to the equation 
$$
\cot 2\theta = \frac{a_{ii}-a_{jj}}{2a_{ij}}.
\tag{a1}
$$
## Zeroing chosen elements
To begin with, we will show that $b_{ij}=0$. Let $B' = U^TX, c=\cos\theta$ and $s=\sin\theta$, then
$$
b'_{ij} = cx_{i_j} + sx_{jj}.
\tag{a2}
$$
Next, let $X=AU$, such that
$$
x_{ij} = -sa_{ii}+ca_{ij}\\
x_{jj} = -sa_{ji}+ca_{jj}.
\tag{a3}
$$
Combining (a2) and (a3), recalling that $A$ is symmetrical s.t. $a_{ij}=a_{ji}$, we get
$$
\begin{aligned}
b_{ij} &= c(-sa_{ii}+ca_{ij}) + s(-sa_{ji}+ca_{jj}) \\
&= c^2a_{ij} + cs(a_{jj}-a_{ii})-s^2a{ji} \\
&= a_{ij}(c^2-s^2)+cs(a_{jj}-a_{ii}),
\end{aligned}
$$
Using the double angle theorems $(c^2+s^2) = \cos 2\theta$ and $cs=(\sin 2\theta)/2$ on the above equation yields
$$
b_{ij} = a_{ij}\cos 2\theta + \frac{1}{2}\sin 2\theta(a_{jj}-a_{ii}).
\tag{a4}
$$
Next, we recall (a1) and rearrange,
$$
\begin{aligned}
\cot 2\theta = \frac{a_{ii}-a_{jj}}{2a_{ij}} &= \frac{\cos 2\theta}{\sin 2\theta} \\
(a_{ii}-a_{jj})\sin 2\theta &= 2a_{ij}\cos 2\theta \\
-\frac{1}{2}(a_{jj}-a_{ii})\sin 2\theta &= a_{ij}\cos 2\theta \\
0 &= a_{ij}\cos 2\theta + \frac{1}{2}(a_{jj}-a_{ii})\sin 2\theta = b_{ij}
\end{aligned}
$$

## Orthonormal basis change preserves symmetry
It is easily seen that $B$ is also symmetric. Simply take the transpose such that 
$$
B^T = (U^TAU)^T = U^T(U^TA)^T = U^TAU = B,
$$
and thus $b_{ij} = b_{ji} = 0$.

The code below generates a random symmetric matrix and displays the zeroed elements $b_{ij}$ and $b_{ji}$.
$i$ and $j$ are chosen such that they correspond to the off diagonal element(s) of maximum absolute value.

In [2]:
# First, we create a symmetric matrix.
A = rand.integers(-5,5,(MSIZE, MSIZE))
A = A.T@A
print("A:")
display(A)

# Some helper functions
def get_theta(aii:float, ajj:float, aij:float):
    # Constructs theta according to (a1)
    return np.atan2(2*aij, aii-ajj)/2

def plane_rotation_matrix(n:np.int64, i:np.int64, j:np.int64, theta:float):
    """Constructs a plane rotation matrix that rotates the
    i:th and j:th axis counter-clockwise by theta.

    Arguments:
        n -- rotation dimension
        i -- i:th axis
        j -- j:th axis
        theta -- rotation angle

    Returns:
        U: U(theta, i, j): NDArray
    """
    U = np.eye(n)
    U[i,i] = np.cos(theta)
    U[j,j] = np.cos(theta)
    U[i,j] = -np.sin(theta)
    U[j,i] = np.sin(theta)
    return U

def get_ij_argmax(A:npt.NDArray):
    """Finds the i,j indices of the element corresponding to the largest
    magnitude in the symmetric matrix A, excluding diagonal."""
    At = np.abs(np.triu(A)) - np.diag(np.diag(np.abs(np.triu(A))))
    i,j = np.unravel_index(At.argmax(), At.shape)
    return i,j

# 
i, j = get_ij_argmax(A)
print(f"The maximum magnitude element of A is A_{i},{j}={A[i,j]} (excluding diagonal).")
theta = get_theta(A[i,i], A[j,j], A[i,j])
U = plane_rotation_matrix(MSIZE, i, j, theta)
print("U:")
display(U)
B = U.T@A@U
print("B:")
display(B)
print(f"b_{i},{j}={B[i,j]:.3f}=b_{j},{i}=b_{B[j,i]:.3f}")

A:


array([[ 48,  25,  22,  27,  -3],
       [ 25,  52,  28,   6,  -3],
       [ 22,  28,  26,  15, -17],
       [ 27,   6,  15,  51,   8],
       [ -3,  -3, -17,   8,  49]])

The maximum magnitude element of A is A_1,2=28 (excluding diagonal).
U:


array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.84294463, -0.53800032,  0.        ,  0.        ],
       [ 0.        ,  0.53800032,  0.84294463,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  1.        ]])

B:


array([[ 4.80000000e+01,  3.29096229e+01,  5.09477385e+00,
         2.70000000e+01, -3.00000000e+00],
       [ 3.29096229e+01,  6.98706981e+01,  7.10542736e-15,
         1.31276726e+01, -1.16748394e+01],
       [ 5.09477385e+00,  5.77315973e-15,  8.12930192e+00,
         9.41616755e+00, -1.27160578e+01],
       [ 2.70000000e+01,  1.31276726e+01,  9.41616755e+00,
         5.10000000e+01,  8.00000000e+00],
       [-3.00000000e+00, -1.16748394e+01, -1.27160578e+01,
         8.00000000e+00,  4.90000000e+01]])

b_1,2=0.000=b_2,1=b_0.000


## Norm preservation and diagonal squared sum

Since $U$ is an orthonormal matrix, it preserves norms. Thus we have that
$$
\sum_{p,q=1}^n |b_{pq}|^2 = \|B\|_F^2 = \|U^TAU\|_F^2 = \|A\|_F^2 = \sum_{p,q=1}^n |a_{pq}|^2,
$$
where $\|\cdot\|_F$ is the Frobenius norm.

By the same method employed in (a2) and (a3), we have the following relations for the upper off diagonal elements of $B$
(with equalities for the corresponding lower off diagonal elements),
$$
\begin{aligned}
b_{pq} &= a_{pq} \\
b_{i,q} &= ca_{iq}-sa_{jq}\\
b_{j,q} &= ca_{jq}+sa_{iq}\\
b_{i,j} &= 0 \\
p,q &\neq i,j.
\end{aligned}
$$
Taking the sum of squares of the off diagonal elements of $B$ we have
$$
\begin{aligned}
&b_{pq}^2 + b_{i,q}^2 + b_{j,q}^2 + b_{i,j}^2 \\
=& a_{pq}^2 + c^2a_{iq}^2 - 2csa_{jq}a_{iq} +s^2a_{jq}^2 + c^2a_{jq}^2+2csa_{jq}a_{iq}+s^2a_{iq}^2+0 \\
=& a_{pq}^2 + a_{iq}^2(c^2+s^2) + a_{jq}^2(c^2+s^2) + csa_{jq}a_{iq}(2-2) \\
=& a_{pq}^2 + a_{iq}^2 + a_{jq}^2 < a_{pq}^2 + a_{iq}^2 + a_{jq}^2 + a_{ij}^2
\end{aligned}
$$
Comparing the squared sums of the upper off diagonal, the term $a_{ij}^2$ vanishes from the upper off diagonal in $B$, as opposed to $A$. The squared sum of the upper off diagonal off $B$ is thus smaller than the corresponding for $A$, or in other words,
$$
\sum_{p\neq q} |b_{pq}|^2 < \sum_{p\neq q} |a_{pq}|^2.
$$
Since the transformation from $B$ to $A$ preserves norm, all energy lost in the off diagonal in $B$ must thus have moved to the diagonal in $B$.

## Converging to eigenvalues
By repeatedly zeroing the maximum magnitude off diagonal element of $A$ by the method described above, one will eventually obtain a diagonal matrix. With some slight change in notation, let the first step of the Jacobi algorithm be
$$
    B_1 = U_1^TAU_1,
$$
where $U_1$ is chosen as described in the prequel to move the energy from the maximum magnitude off diagonal element of $A$ into the diagonal of $B$. Repeating this procedure, in step 2 we will have
$$
B_2 = U_2^TB_1U_2 = U_2^TU_1^TAU_1U_2.
$$
This is then repeated until we obtain a diagonal matrix $A_n$ as below
$$
\begin{aligned}
B_n &= U_n^TB_{n-1}U_n = E^TAE \\
E &= U_1U_2\cdots U_n.
\end{aligned}
$$
Since the transforms $U_k^TA_kU_k$ preserves the eigenvalues of $A$, they are easily found as the diagonal elements of $B_n$. In fact, we have obtained an eigenvalue similarity transform, where the rows of $E$ are the eigenvectors of $A$.

The code below demonstrates the algorithm, and compares with numpy's built in algorithms.

In [3]:
# Helper function to stop the algorithm.
def is_diagonal(M:npt.NDArray):
    #First check that M is symmetric.
    if not np.allclose(M,M.T): return False
    return np.allclose(M-np.diag(np.diag(M)), np.zeros_like(M))

E = U
Un = U
Bn = B
while not is_diagonal(Bn):
    i, j = get_ij_argmax(Bn)
    theta = get_theta(Bn[i,i], Bn[j,j], Bn[i,j])
    Un = plane_rotation_matrix(MSIZE, i, j, theta)
    Bn = Un.T@Bn@Un

    E = E@Un

# Need to make sure the eigs are sorted in the same order
eig_vals_np, eig_vecs_np = np.linalg.eig(A)
idx = np.flip(eig_vals_np.argsort())
eig_vals_np = eig_vals_np[idx]
eig_vecs_np = eig_vecs_np[:,idx]

idx = np.flip(np.diag(Bn).argsort())
eig_vals = np.diag(Bn)[idx]
eig_vecs = E[:,idx]


print(f"Eigenvalues of A(numpy):")
display(eig_vals_np)
print(f"Eigenvalues of B_n:")
display(eig_vals)

print(f"Eigenvectors of A(numpy):")
display(eig_vecs_np)
print(f"E:")
display(eig_vecs)

Eigenvalues of A(numpy):


array([108.12695339,  60.28785613,  39.98875591,  16.34866432,
         1.24777026])

Eigenvalues of B_n:


array([108.12695339,  60.28785613,  39.98875591,  16.34866432,
         1.24777026])

Eigenvectors of A(numpy):


array([[-0.57704788, -0.1424656 ,  0.09236132, -0.79874844, -0.01377001],
       [-0.52768805,  0.2274995 , -0.65897858,  0.27138292, -0.40235288],
       [-0.43799095,  0.23066895,  0.03472943,  0.26504312,  0.82674166],
       [-0.42574047, -0.5476759 ,  0.49222821,  0.46636159, -0.24292883],
       [ 0.12437864, -0.75814937, -0.56010881, -0.02472353,  0.30887919]])

E:


array([[ 0.57704788, -0.1424656 ,  0.09236132, -0.79874844, -0.01377001],
       [ 0.52768805,  0.2274995 , -0.65897858,  0.27138292, -0.40235288],
       [ 0.43799095,  0.23066895,  0.03472943,  0.26504312,  0.82674166],
       [ 0.42574047, -0.5476759 ,  0.49222821,  0.46636159, -0.24292883],
       [-0.12437864, -0.75814937, -0.56010881, -0.02472353,  0.30887919]])